## User Input

In [ ]:
#Ask user to establish hotspot search distance

import arcpy
import os


#Set up Workspace
workspace = r"<workspace_path>"
arcpy.env.workspace = workspace
arcpy.env.overwriteOutput = True


#User Input
search_distance = float(input("Enter hotspot distance band in meters: ")) #meters
burn_threshold = float(input("Enter burn threshold in acres: ")) #acres
patch_threshold = float(input("Enter patch threshold in acres: ")) #acres


#Input Layers
#NOTE: the feature class "Caribou_WinterRange__Project_NAD1983Albers" 
#contains TWO underscores between WinterRange and Project. 
#This was an accidental naming issue during project development, but
#the layer has already been used in multiple workflows.
#Do not rename unless all references are updated.
winter_points = "Caribou_WinterRange__Project_NAD1983Albers"
spruce_lichen = "SpruceLichen_Simplify_Polygon"
fire_layer = "AK_FireAcreage_WinterRange_Clip"

#Output Layers
fishnet = "WinterRange_Fishnet_1Km"
spatial_join = "WinterRange_SpatialJoin"
hotspots = f"WinterRange_Hotspots_{int(search_distance)}m"

## Fishnet Creation

In [ ]:
#Create a fishnet by first using describe and extent for winter_points layer

#Describe the projected winter caribou GPS layer.

#This will tell ArcPy to look at the layer and make its 
#properties available for use in the script.
desc = arcpy.Describe(winter_points)

#Obtain the spatial extent of the GPS points.

#The extent will store the min and max X and Y coordinates (bounding box)
#that will then be used to determine the area covered by the fishnet.
extent = desc.extent

#Store the lower-left corner coordinates.
origin = f"{extent.XMin} {extent.YMin}"

#Store the upper-right corner coordinates.
corner = f"{extent.XMax} {extent.YMax}"

In [ ]:
#Create a 1km x 1km fishnet covering the extent of the projected winter caribou GPS Observations

#fishnet = output feature class name
#origin = lower-left corner of the study area extent
#Y-axis coordinate = establishes grid orientation. ArcGIS requires a Y-axis coordinate even when a template extent is supplied. A point 10 units above the origin is used to establish the grid orientation.
#1000, 1000 = 1 km cell width and heighth
#"" = allow ArcGIS to automatically calculate rows
#"" = allow ArcGIS to automatically calculate columns
#corner = upper-right corner of the study area extent
#NO_LABELS = do not create a label point feature class
#winter_points = use GPS layer as template extent
#POLYGON = create polygon fishnet cells
arcpy.management.CreateFishnet(fishnet, origin, f"{extent.XMin} {extent.YMin + 10}", 1000, 1000, "", "", corner, "NO_LABELS", winter_points, "POLYGON")

# Spatial Join

In [ ]:
#Spatially join winter_points layer to the newly created fishnet grid layer

#Each fishnet cell will receive a count of how many GPS observations fall inside that cell.

#These counts will later be used as the input field for the Getis-Ord Gi* hotspot Analysis.

#Target layer: fishnet, Join features: winter_points, Output: spatial_join (variable of WinterRange_SpatialJoin above)

arcpy.analysis.SpatialJoin(fishnet, winter_points, spatial_join, "JOIN_ONE_TO_ONE", "KEEP_ALL")

In [ ]:
#confirm output exists

print(arcpy.Exists(spatial_join))

## Hotspot Analysis (Getis-Ord Gi*)

In [ ]:
#Identify statistically significant clusters of high and low winter GPS observation counts
#using the Getis-Ord Gi* statistic.

#Input Features: spatial_join
#Analysis Field: Join_Count
#Distance Band: user-defined search_distance
#Output: hotspots

#FIXED_DISTANCE_BAND: each fishnet cell is compared to neighboring cells within a user-defined search distance.
#EUCLIDEAN_DISTANCE: distance is measured as a straight line distance between cell centers.
#NONE: 
#search_distance: user-defined neighborhood distance used to determine which cells are considered neighbors.
arcpy.stats.HotSpots(spatial_join, "Join_Count", hotspots, "FIXED_DISTANCE_BAND", "EUCLIDEAN_DISTANCE", "NONE", search_distance)

## Fire: 1st Loop (Year) and 1st Conditional (Acreage)

In [ ]:
#Inspect fire layer fields

fire_layer = "AK_FireAcreage_WinterRange_Clip"

for field in arcpy.ListFields(fire_layer):
    print(field.name)



In [ ]:
#Create a list of fire years

fire_layer = "AK_FireAcreage_WinterRange_Clip"

fire_years = sorted(list(set(row[0] for row in arcpy.da.SearchCursor(fire_layer, ["FIREYEAR"]))))

print(fire_years)

In [ ]:
#Loop through fire years

for year in fire_years:
    print(f"Processing Fire Year: {year}")

In [ ]:
#Select fires one year at a time

#Fire year loop
#Iterates through each fire year and selects fire number of polygons for year-specific analysis.
#Also selects fires that meet or exceed the user-defined acreage threshold.

for year in fire_years:
    where_clause = f"FIREYEAR = '{year}' AND ACRES >= {burn_threshold}"

    arcpy.management.MakeFeatureLayer(fire_layer, "yearly_fires", where_clause)

    count = int(arcpy.management.GetCount("yearly_fires")[0])

    print(f"{year}: {count} fire polygons")

## Vegetation: 2nd Loop (vegetation class) and 2nd Conditional (habitat patch size)

In [ ]:
#Run the following to find the vegetation layer name needed

veg_layer = "SpruceLichen_Simplify_Polygon"

for field in arcpy.ListFields(veg_layer):
    print(field.name)

In [ ]:
#Create a list of vegetation classes

vegetation_classes = sorted(list(set(row[0] for row in arcpy.da.SearchCursor(veg_layer, ["EVT_NAME"]))))

print(vegetation_classes)

In [ ]:
#Vegetation Class Loop

for veg_class in vegetation_classes:
    print(veg_class)

In [ ]:
#Vegetation Class Loop & Habitat Patch Size Condition

for veg_class in vegetation_classes:
    where_clause = (f"EVT_NAME = '{veg_class}' AND Shape_Area >= {patch_threshold}")

    arcpy.management.MakeFeatureLayer(veg_layer, "veg_patches", where_clause)

    count = int(arcpy.management.GetCount("veg_patches")[0])

    print(f"{veg_class}: {count} habitat patches")